In [11]:
import polars as pl

# Show all rows and columns

pl.Config(tbl_rows=-1, tbl_cols=-1)

# Read CSV data as a dataframe

games = pl.read_csv("games.csv")

print(games)

shape: (272, 4)
┌───────────────────────┬────────────┬───────────────────────┬────────────┐
│ away_team             ┆ away_score ┆ home_team             ┆ home_score │
│ ---                   ┆ ---        ┆ ---                   ┆ ---        │
│ str                   ┆ i64        ┆ str                   ┆ i64        │
╞═══════════════════════╪════════════╪═══════════════════════╪════════════╡
│ Dallas Cowboys        ┆ 20         ┆ Philadelphia Eagles   ┆ 24         │
│ Kansas City Chiefs    ┆ 21         ┆ Los Angeles Chargers  ┆ 27         │
│ Arizona Cardinals     ┆ 20         ┆ New Orleans Saints    ┆ 13         │
│ Pittsburgh Steelers   ┆ 34         ┆ New York Jets         ┆ 32         │
│ Miami Dolphins        ┆ 8          ┆ Indianapolis Colts    ┆ 33         │
│ Tampa Bay Buccaneers  ┆ 23         ┆ Atlanta Falcons       ┆ 20         │
│ New York Giants       ┆ 6          ┆ Washington Commanders ┆ 21         │
│ Carolina Panthers     ┆ 10         ┆ Jacksonville Jaguars  ┆ 26       

In [9]:
# Separate each game into two entries, one for the home team and one for away

homeResults = games.select(
    pl.col("home_team").alias("team"),
    pl.col("home_score").alias("scored"),
    pl.col("away_score").alias("allowed"),
)

awayResults = games.select(
    pl.col("away_team").alias("team"),
    pl.col("away_score").alias("scored"),
    pl.col("home_score").alias("allowed"),
)

# Combine the home and away win loss results

combined = pl.concat([homeResults, awayResults])

print(combined)

shape: (544, 3)
┌───────────────────────┬────────┬─────────┐
│ team                  ┆ scored ┆ allowed │
│ ---                   ┆ ---    ┆ ---     │
│ str                   ┆ i64    ┆ i64     │
╞═══════════════════════╪════════╪═════════╡
│ Philadelphia Eagles   ┆ 24     ┆ 20      │
│ Los Angeles Chargers  ┆ 27     ┆ 21      │
│ New Orleans Saints    ┆ 13     ┆ 20      │
│ New York Jets         ┆ 32     ┆ 34      │
│ Indianapolis Colts    ┆ 33     ┆ 8       │
│ Atlanta Falcons       ┆ 20     ┆ 23      │
│ Washington Commanders ┆ 21     ┆ 6       │
│ Jacksonville Jaguars  ┆ 26     ┆ 10      │
│ Cleveland Browns      ┆ 16     ┆ 17      │
│ New England Patriots  ┆ 13     ┆ 20      │
│ Seattle Seahawks      ┆ 13     ┆ 17      │
│ Denver Broncos        ┆ 20     ┆ 12      │
│ Los Angeles Rams      ┆ 14     ┆ 9       │
│ Green Bay Packers     ┆ 27     ┆ 13      │
│ Buffalo Bills         ┆ 41     ┆ 40      │
│ Chicago Bears         ┆ 24     ┆ 27      │
│ Green Bay Packers     ┆ 27     ┆ 18  

In [12]:
# Group the stats from each team game into a single row

grouped = combined.group_by("team").agg(
    pl.len().alias("numGames"),
    (pl.col("scored") > pl.col("allowed")).sum().alias("wins"),
    (pl.col("scored") < pl.col("allowed")).sum().alias("losses"),
    (pl.col("scored") == pl.col("allowed")).sum().alias("ties"),
    pl.sum("scored"),
    pl.sum("allowed"),
)

print(grouped)

shape: (32, 7)
┌───────────────────────┬──────────┬──────┬────────┬──────┬────────┬─────────┐
│ team                  ┆ numGames ┆ wins ┆ losses ┆ ties ┆ scored ┆ allowed │
│ ---                   ┆ ---      ┆ ---  ┆ ---    ┆ ---  ┆ ---    ┆ ---     │
│ str                   ┆ u32      ┆ u32  ┆ u32    ┆ u32  ┆ i64    ┆ i64     │
╞═══════════════════════╪══════════╪══════╪════════╪══════╪════════╪═════════╡
│ Atlanta Falcons       ┆ 17       ┆ 8    ┆ 9      ┆ 0    ┆ 353    ┆ 401     │
│ Buffalo Bills         ┆ 17       ┆ 12   ┆ 5      ┆ 0    ┆ 481    ┆ 365     │
│ Tampa Bay Buccaneers  ┆ 17       ┆ 8    ┆ 9      ┆ 0    ┆ 380    ┆ 411     │
│ Kansas City Chiefs    ┆ 17       ┆ 6    ┆ 11     ┆ 0    ┆ 362    ┆ 328     │
│ Cincinnati Bengals    ┆ 17       ┆ 6    ┆ 11     ┆ 0    ┆ 414    ┆ 492     │
│ Washington Commanders ┆ 17       ┆ 5    ┆ 12     ┆ 0    ┆ 356    ┆ 451     │
│ Los Angeles Chargers  ┆ 17       ┆ 11   ┆ 6      ┆ 0    ┆ 368    ┆ 340     │
│ Tennessee Titans      ┆ 17       ┆ 

In [13]:
# Calculate the per-game stats for each team
# Sort the team data by win percentage, highest to lowest

finalStats = grouped.with_columns(
    ((pl.col("wins") + 0.5 * pl.col("ties")) / pl.col("numGames")).round(3).alias("winPct"),
    (pl.col("scored") / pl.col("numGames")).round(1).alias("scoredPerGame"),
    (pl.col("allowed") / pl.col("numGames")).round(1).alias("allowedPerGame"),
).sort("winPct", descending=True)

print(finalStats)

shape: (32, 10)
┌───────────────────────┬──────────┬──────┬────────┬──────┬────────┬─────────┬────────┬───────────────┬────────────────┐
│ team                  ┆ numGames ┆ wins ┆ losses ┆ ties ┆ scored ┆ allowed ┆ winPct ┆ scoredPerGame ┆ allowedPerGame │
│ ---                   ┆ ---      ┆ ---  ┆ ---    ┆ ---  ┆ ---    ┆ ---     ┆ ---    ┆ ---           ┆ ---            │
│ str                   ┆ u32      ┆ u32  ┆ u32    ┆ u32  ┆ i64    ┆ i64     ┆ f64    ┆ f64           ┆ f64            │
╞═══════════════════════╪══════════╪══════╪════════╪══════╪════════╪═════════╪════════╪═══════════════╪════════════════╡
│ Seattle Seahawks      ┆ 17       ┆ 14   ┆ 3      ┆ 0    ┆ 483    ┆ 292     ┆ 0.824  ┆ 28.4          ┆ 17.2           │
│ Denver Broncos        ┆ 17       ┆ 14   ┆ 3      ┆ 0    ┆ 401    ┆ 311     ┆ 0.824  ┆ 23.6          ┆ 18.3           │
│ New England Patriots  ┆ 17       ┆ 14   ┆ 3      ┆ 0    ┆ 490    ┆ 320     ┆ 0.824  ┆ 28.8          ┆ 18.8           │
│ Jacksonville J